In [ ]:
# Install required packages if they are missing
required_packages <- c("cellWise", "robustHD", "ggplot2", "ggrepel",
                       "dplyr", "gridExtra", "tidyr", "readr")
new_packages <- required_packages[!(required_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)


In [5]:
list.files("/kaggle/input/", recursive = TRUE)

[1] "datasets/giljorge/copy-world-bank-data/world_bank_development_indicators.csv"

In [ ]:
# =============================================================================
# 01_simulation_study.R
#
# Partial replication of the simulation study from:
#   Hubert et al. (2019) MacroPCA, Technometrics 61, 459-473.
#
# We replicate:
#   - Figure 7: 20% NAs + 20% cellwise outliers  (MSE vs gamma)
#   - Figure 9: 20% NAs + 10% cellwise + 10% rowwise outliers (MSE vs gamma)
#
# Methods compared: ICPCA, MROBPCA, MacroPCA  (as in the paper)
# Metric: MSE against baseline PCA on clean data  (paper Section 5)
# Data generating process: A09 covariance, n=100, d=200, k=6  (paper Section 5)
#
# NOTE: With d=200 this takes ~20-40 min. Reduce n_sim or d to prototype faster.
# =============================================================================

library(cellWise)   # MacroPCA, ICPCA, MROBPCA, DDC
library(MASS)       # mvrnorm (backup)
library(ggplot2)
library(dplyr)
library(tidyr)

set.seed(2024)

# =============================================================================
# 1. Data generating process  (Section 5 of the paper)
# =============================================================================

n      <- 100   # observations
d      <- 200   # variables
k      <- 6     # true number of components
n_sim  <- 30    # Monte Carlo replications (paper uses 100; reduce for speed)

cat("Building A09 covariance matrix (d =", d, ")...\n")

# A09 structured correlation: rho_{ij} = (-0.9)^|i-j|
idx    <- matrix(1:d, d, d)
R_A09  <- (-0.9)^abs(idx - t(idx))

# Target eigenvalues: 6 large + 194 small (paper Section 5)
lambda_large  <- c(30, 25, 20, 15, 10, 5)
lambda_small  <- seq(0.098, 0.0015, length.out = d - k)
lambda_target <- c(lambda_large, lambda_small)

# Build Sigma by replacing eigenvalues of R_A09
eig_R     <- eigen(R_A09, symmetric = TRUE)
Sigma     <- eig_R$vectors %*% diag(lambda_target) %*% t(eig_R$vectors)
Sigma     <- (Sigma + t(Sigma)) / 2          # ensure symmetry

# Variables for contamination
sigma_j   <- sqrt(diag(Sigma))               # column SDs, used for cellwise shift
v_kp1     <- eig_R$vectors[, k + 1]          # (k+1)-th eigenvector, for rowwise shift

# Fast clean-data generator using Cholesky
chol_Sig  <- chol(Sigma)                     # upper-triangular
generate_clean <- function(n) {
  matrix(rnorm(n * d), n, d) %*% chol_Sig   # n x d
}

cat("Done. Sigma built.\n")

# =============================================================================
# 2. MSE helper
#
# Baseline: classical PCA on the CLEAN rows of the uncontaminated data X0.
# For each method applied to contaminated data, compute predictions for those
# same clean rows and measure MSE against the baseline predictions.
# Paper eq: MSE = (1/cd) * sum_{i in C} sum_j (xhat_ij - xhat^C_ij)^2
# =============================================================================
compute_predictions <- function(center, loadings, X_data) {
  # X_data: n x d (may contain NAs; missing entries get predicted from subspace)
  # Returns n x d matrix of predicted values
  X_c  <- sweep(X_data, 2, center, "-")      # centre
  # For rows with NAs, use only observed entries to compute scores
  scores <- matrix(NA, nrow(X_data), ncol(loadings))
  for (i in seq_len(nrow(X_data))) {
    obs <- !is.na(X_c[i, ])
    if (sum(obs) >= ncol(loadings)) {
      # Least-squares projection onto observed dimensions
      P_obs    <- loadings[obs, , drop = FALSE]
      scores[i, ] <- solve(t(P_obs) %*% P_obs) %*% t(P_obs) %*% X_c[i, obs]
    } else {
      scores[i, ] <- 0
    }
  }
  Xhat <- sweep(scores %*% t(loadings), 2, center, "+")
  Xhat
}

compute_mse <- function(Xhat_method, Xhat_base, C_rows) {
  # MSE over clean rows C and all d columns
  diff  <- Xhat_method[C_rows, ] - Xhat_base[C_rows, ]
  mean(diff^2, na.rm = TRUE)
}

# =============================================================================
# 3. Contamination functions
# =============================================================================
add_nas <- function(X, frac = 0.20) {
  Xc     <- X
  n_miss <- round(frac * length(X))
  idx    <- sample(length(X), n_miss)
  Xc[idx] <- NA
  Xc
}

add_cellwise <- function(X, frac = 0.20, gamma) {
  Xc     <- X
  n_cont <- round(frac * length(X))
  idx    <- sample(length(X), n_cont)
  # shift by gamma * sigma_j  (paper: replace x_ij with gamma * sigma_j)
  col_idx <- ((idx - 1) %% d) + 1
  Xc[idx] <- gamma * sigma_j[col_idx]
  Xc
}

add_rowwise <- function(X, frac = 0.20, gamma) {
  Xc      <- X
  bad     <- sample(nrow(X), round(frac * nrow(X)))
  # shift from N(gamma * v_{k+1}, Sigma)  (paper Section 5)
  shift   <- gamma * v_kp1                   # d-vector
  for (i in bad) {
    noise      <- matrix(rnorm(d), 1, d) %*% chol_Sig
    Xc[i, ]    <- shift + noise
  }
  list(X = Xc, bad_rows = bad)
}

# =============================================================================
# 4. Run simulation for a given scenario
# Scenario A: 20% NA + 20% cellwise  (Figure 7)
# Scenario B: 20% NA + 10% cellwise + 10% rowwise  (Figure 9)
# =============================================================================
gamma_vals <- c(0, 1, 2, 3, 5, 7, 10, 15, 20)

run_scenario <- function(scenario_name, frac_cell, frac_row) {
  cat("\n=== Scenario:", scenario_name, "===\n")

  results <- expand.grid(
    gamma  = gamma_vals,
    method = c("ICPCA", "MROBPCA", "MacroPCA"),
    mse    = NA_real_,
    stringsAsFactors = FALSE
  )

  for (gi in seq_along(gamma_vals)) {
    gamma <- gamma_vals[gi]
    cat("  gamma =", gamma, "\n")

    mse_icpca    <- numeric(n_sim)
    mse_mrobpca  <- numeric(n_sim)
    mse_macro    <- numeric(n_sim)

    for (s in seq_len(n_sim)) {
      # --- Generate clean data ---
      X0 <- generate_clean(n)

      # --- Baseline: classical PCA on full clean data ---
      pca_base    <- prcomp(X0, center = TRUE, scale. = FALSE)
      center_base <- colMeans(X0)
      scores_base <- X0 %*% pca_base$rotation[, 1:k, drop = FALSE]
      Xhat_base   <- sweep(
        scores_base %*% t(pca_base$rotation[, 1:k, drop = FALSE]),
        2, center_base, "+"
      )

      # --- Contaminate ---
      X_cont  <- X0
      bad_rows <- integer(0)

      if (frac_cell > 0) {
        X_cont <- add_cellwise(X_cont, frac = frac_cell, gamma = gamma)
      }
      if (frac_row > 0) {
        rw       <- add_rowwise(X_cont, frac = frac_row, gamma = gamma)
        X_cont   <- rw$X
        bad_rows <- rw$bad_rows
      }
      # Add NAs last (so we know which cells are outliers vs missing)
      X_cont <- add_nas(X_cont, frac = 0.20)

      # Clean rows for MSE evaluation
      C_rows <- setdiff(seq_len(n), bad_rows)

      # --- ICPCA ---
      fit_icpca <- tryCatch(
        ICPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_icpca)) {
        Xhat_i  <- compute_predictions(fit_icpca$center,
                                       fit_icpca$loadings, X0)
        mse_icpca[s] <- compute_mse(Xhat_i, Xhat_base, C_rows)
      } else {
        mse_icpca[s] <- NA
      }

      # --- MROBPCA ---
      fit_mrob <- tryCatch(
        MROBPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_mrob)) {
        Xhat_m  <- compute_predictions(fit_mrob$center,
                                       fit_mrob$loadings, X0)
        mse_mrobpca[s] <- compute_mse(Xhat_m, Xhat_base, C_rows)
      } else {
        mse_mrobpca[s] <- NA
      }

      # --- MacroPCA ---
      fit_macro <- tryCatch(
        MacroPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_macro)) {
        Xhat_p  <- compute_predictions(fit_macro$center,
                                       fit_macro$loadings, X0)
        mse_macro[s] <- compute_mse(Xhat_p, Xhat_base, C_rows)
      } else {
        mse_macro[s] <- NA
      }
    } # end sim loop

    results$mse[results$gamma == gamma & results$method == "ICPCA"]    <- mean(mse_icpca,   na.rm = TRUE)
    results$mse[results$gamma == gamma & results$method == "MROBPCA"]  <- mean(mse_mrobpca, na.rm = TRUE)
    results$mse[results$gamma == gamma & results$method == "MacroPCA"] <- mean(mse_macro,   na.rm = TRUE)
  } # end gamma loop

  results
}

# Run both scenarios
res_fig7 <- run_scenario("Fig7: 20% NA + 20% cellwise",
                         frac_cell = 0.20, frac_row = 0.00)

res_fig9 <- run_scenario("Fig9: 20% NA + 10% cell + 10% row",
                         frac_cell = 0.10, frac_row = 0.10)

# =============================================================================
# 5. Plots  (line plots of avg MSE vs gamma, one curve per method)
# =============================================================================
method_colors <- c(
  "ICPCA"    = "#E07B54",
  "MROBPCA"  = "#4C8BB5",
  "MacroPCA" = "#2CA02C"
)
method_lines <- c(
  "ICPCA"    = "dashed",
  "MROBPCA"  = "dotted",
  "MacroPCA" = "solid"
)

make_mse_plot <- function(results, title_str, y_lim = NULL) {
  p <- ggplot(results, aes(x = gamma, y = mse,
                           colour = method, linetype = method)) +
    geom_line(linewidth = 0.9) +
    geom_point(size = 2) +
    scale_colour_manual(values = method_colors) +
    scale_linetype_manual(values = method_lines) +
    labs(
      title    = title_str,
      subtitle = paste0("n = ", n, ", d = ", d, ", k = ", k,
                        ", ", n_sim, " replications, A09 covariance"),
      x        = expression(gamma ~ "(contamination distance)"),
      y        = "Average MSE",
      colour   = "Method",
      linetype = "Method"
    ) +
    theme_bw(base_size = 13) +
    theme(legend.position = "bottom")

  if (!is.null(y_lim)) p <- p + coord_cartesian(ylim = y_lim)
  p
}

p_fig7 <- make_mse_plot(
  res_fig7,
  "Figure 7 replication: 20% missing + 20% cellwise outliers"
)

p_fig9 <- make_mse_plot(
  res_fig9,
  "Figure 9 replication: 20% missing + 10% cellwise + 10% rowwise outliers"
)

print(p_fig7)
print(p_fig9)

ggsave("sim_figure7_replication.pdf", p_fig7, width = 7, height = 5)
ggsave("sim_figure9_replication.pdf", p_fig9, width = 7, height = 5)

# =============================================================================
# 6. Combined panel (both scenarios side by side)
# =============================================================================
res_fig7$scenario <- "20% NA + 20% cellwise"
res_fig9$scenario <- "20% NA + 10% cell + 10% row"
res_combined      <- rbind(res_fig7, res_fig9)

p_combined <- ggplot(res_combined,
                     aes(x = gamma, y = mse,
                         colour = method, linetype = method)) +
  geom_line(linewidth = 0.9) +
  geom_point(size = 1.8) +
  scale_colour_manual(values = method_colors) +
  scale_linetype_manual(values = method_lines) +
  facet_wrap(~ scenario, scales = "free_y") +
  labs(
    title    = "Simulation study: ICPCA vs MROBPCA vs MacroPCA",
    subtitle = paste0("A09 covariance, n = ", n, ", d = ", d,
                      ", k = ", k, ", ", n_sim, " MC replications"),
    x        = expression(gamma),
    y        = "Average MSE",
    colour   = "Method",
    linetype = "Method"
  ) +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom")

print(p_combined)
ggsave("sim_combined_panel.pdf", p_combined, width = 11, height = 5)

cat("\n=== Simulation study complete. Saved three PDFs. ===\n")

# Print summary tables
cat("\n--- Figure 7 results (avg MSE) ---\n")
print(pivot_wider(res_fig7[, c("gamma","method","mse")],
                  names_from = method, values_from = mse))

cat("\n--- Figure 9 results (avg MSE) ---\n")
print(pivot_wider(res_fig9[, c("gamma","method","mse")],
                  names_from = method, values_from = mse))


In [ ]:
# =============================================================================
# 02_real_data_analysis.R
#
# Application of MacroPCA to Global Economic Indicators (2024 cross-section)
# Mirrors the structure of Hubert et al. (2019) Section 3:
#   - Residual maps, outlier maps, loadings comparison, online prediction
#
# Dataset: Global Economic Indicators 2010-2025
#   https://www.kaggle.com/datasets/tanishksharma9905/global-economic-indicators-20102025
# =============================================================================

library(cellWise)
library(ggplot2)
library(ggrepel)
library(dplyr)
library(tidyr)
library(gridExtra)
library(readr)

# =============================================================================
# 1. Load data
# =============================================================================
econ_raw <- NULL
csv_paths <- c(
  "/kaggle/input/datasets/giljorge/copy-world-bank-data/world_bank_development_indicators.csv"
)
for (p in csv_paths) {
  if (file.exists(p)) { econ_raw <- read_csv(p, show_col_types=FALSE); cat("Loaded:", p, "\n"); break }
}
if (is.null(econ_raw)) stop("Dataset file not found.")

cat("Raw dimensions:", nrow(econ_raw), "x", ncol(econ_raw), "\n")

# Identify country column
country_col <- colnames(econ_raw)[grepl("(?i)country", colnames(econ_raw), perl=TRUE)][1]
cat("Country column:", country_col, "\n\n")

# =============================================================================
# 2. One row per country: take the most recent non-NA values across all years
#    (avoids the broken year-filter that left 0 usable columns)
# =============================================================================
# Identify numeric indicator columns — exclude id/date/year/country/text cols
exclude_pat <- "(?i)(year|date|country|iso|code|region|income|name)"
num_cols <- colnames(econ_raw)[sapply(econ_raw, is.numeric)]
num_cols <- num_cols[!grepl(exclude_pat, num_cols)]
cat("Numeric indicator columns:", length(num_cols), "\n")

# For each country, take the mean across years (robust to sparse years)
econ_agg <- econ_raw %>%
  group_by(!!sym(country_col)) %>%
  summarise(across(all_of(num_cols), ~ mean(.x, na.rm=TRUE)), .groups="drop")

# Convert NaN (all-NA groups) back to NA
econ_agg[num_cols] <- lapply(econ_agg[num_cols], function(x) ifelse(is.nan(x), NA, x))

cat("Aggregated countries:", nrow(econ_agg), "\n")

# Keep columns with <50% missing
na_frac   <- sapply(econ_agg[num_cols], function(x) mean(is.na(x)))
keep_cols <- num_cols[na_frac < 0.5]
cat("Columns with <50% NA:", length(keep_cols), "\n")

if (length(keep_cols) < 2) stop("Fewer than 2 usable columns after NA filter. Check dataset.")

# Build matrix
X_raw <- data.matrix(econ_agg[, keep_cols, drop=FALSE])
rownames(X_raw) <- make.unique(as.character(econ_agg[[country_col]]))

# Keep rows with <50% missing
X_raw <- X_raw[rowMeans(is.na(X_raw)) < 0.5, ]
obs_names <- rownames(X_raw)

cat("Final:", nrow(X_raw), "countries x", ncol(X_raw), "indicators\n")
cat("Missing cells:", sum(is.na(X_raw)),
    sprintf("(%.1f%%)\n\n", 100*mean(is.na(X_raw))))

# =============================================================================
# 3. Log-transform right-skewed variables (GDP, trade flows, etc.)
# =============================================================================
log_candidates <- colnames(X_raw)[grepl("(?i)(gdp|trade|export|import|debt|population|fdi)",
                                         colnames(X_raw), perl=TRUE)]

X <- X_raw
log_vars_used <- c()
for (v in log_candidates) {
  col <- X_raw[, v]
  if (all(col[!is.na(col)] > 0)) {
    X[, v] <- log(col)
    log_vars_used <- c(log_vars_used, v)
  }
}
X[!is.finite(X)] <- NA
if (length(log_vars_used) > 0) {
  cat("Log-transformed:", paste(log_vars_used, collapse=", "), "\n\n")
} else {
  cat("No log-transforms applied (no all-positive skewed columns found)\n\n")
}

# =============================================================================
# 4. Fit MacroPCA
# =============================================================================
k <- 2

cat("Fitting MacroPCA (k =", k, ")...\n")
fit_macro <- MacroPCA(X, k = k)

# Build indcells as proper n x p matrix from stdResid
macro_ind_mat <- matrix(0L, nrow(fit_macro$stdResid), ncol(fit_macro$stdResid))
macro_ind_mat[!is.na(fit_macro$stdResid) & fit_macro$stdResid >  2.576] <-  1L
macro_ind_mat[!is.na(fit_macro$stdResid) & fit_macro$stdResid < -2.576] <- -1L
fit_macro$indcells <- macro_ind_mat

# Align: MacroPCA drops rows with >50% NAs
kept_rows <- match(rownames(fit_macro$stdResid), rownames(X))
obs_names <- obs_names[kept_rows]
X_with_na <- X[kept_rows, ]   # keep NA version for ICPCA residual map
X         <- X[kept_rows, ]

# Impute X (column medians) — needed for ICPCA and train/test split
for (j in seq_len(ncol(X))) {
  col_med <- median(X[, j], na.rm = TRUE)
  if (!is.finite(col_med)) col_med <- 0
  X[is.na(X[, j]), j] <- col_med
}

cat("MacroPCA: Cumulative variance explained:",
    round(cumsum(fit_macro$eigenvalues /
                   sum(fit_macro$eigenvalues))[1:k], 3), "\n")
cat("MacroPCA: Flagged cellwise outliers:",
    sum(fit_macro$indcells != 0, na.rm = TRUE), "\n")
cat("MacroPCA: Flagged casewise outliers:",
    sum(fit_macro$indrows, na.rm = TRUE), "\n\n")

# =============================================================================
# 5. Fit ICPCA
# =============================================================================
cat("Fitting ICPCA (k =", k, ")...\n")
set.seed(42)
X_icpca <- X
for (j in seq_len(ncol(X_icpca))) {
  if (sd(X_icpca[, j]) < 1e-10)
    X_icpca[, j] <- X_icpca[, j] + rnorm(nrow(X_icpca), 0, 1e-6)
}
fit_icpca <- tryCatch(
  ICPCA(X_icpca, k = k),
  error = function(e) { message("ICPCA failed: ", conditionMessage(e)); NULL }
)
if (is.null(fit_icpca)) stop("ICPCA could not be fitted.")

cat("ICPCA: Cumulative variance explained:",
    round(cumsum(fit_icpca$eigenvalues /
                   sum(fit_icpca$eigenvalues))[1:k], 3), "\n\n")

# =============================================================================
# 6. Select representative countries for residual map
# =============================================================================
# Top OD countries + any notable ones present in the data
notable <- c("United States", "China", "Germany", "Japan", "Brazil",
             "India", "Russian Federation", "South Africa", "Nigeria",
             "Venezuela, RB", "Zimbabwe", "Afghanistan")
note_idx <- which(obs_names %in% notable)
top_od   <- order(fit_macro$OD, decreasing = TRUE)[1:10]
sel_rows <- unique(c(note_idx, top_od))
sel_rows <- sel_rows[seq_len(min(24, length(sel_rows)))]
sel_names <- obs_names[sel_rows]

cat("Selected", length(sel_rows), "countries for residual map.\n")

# =============================================================================
# 7. Residual maps
# =============================================================================
pdf("residual_map_macropca.pdf", width = 12, height = 7)
cellMap(
  fit_macro$stdResid[sel_rows, ],
  indcells     = which((fit_macro$indcells != 0)[sel_rows, ]),
  rowlabels    = sel_names,
  columnlabels = colnames(X),
  mTitle       = "MacroPCA \u2013 Residual map"
)
dev.off()
cat("Saved: residual_map_macropca.pdf\n")

# ICPCA residual map
Xhat_icpca <- sweep(
  fit_icpca$scores %*% t(fit_icpca$loadings),
  2, fit_icpca$center, "+"
)
X_naimp_icpca <- X_with_na
for (j in seq_len(ncol(X_with_na))) {
  na_j <- is.na(X_with_na[, j])
  X_naimp_icpca[na_j, j] <- Xhat_icpca[na_j, j]
}
resid_icpca <- X_naimp_icpca - Xhat_icpca
col_mad     <- apply(resid_icpca, 2, function(x) mad(x, na.rm = TRUE))
col_mad[col_mad < 1e-10] <- 1
stdR_icpca  <- sweep(resid_icpca, 2, col_mad, "/")
indcells_icpca <- matrix(0L, nrow(X), ncol(X))
indcells_icpca[!is.na(stdR_icpca) & stdR_icpca >  2.576] <-  1L
indcells_icpca[!is.na(stdR_icpca) & stdR_icpca < -2.576] <- -1L

pdf("residual_map_icpca.pdf", width = 12, height = 7)
cellMap(
  stdR_icpca[sel_rows, ],
  indcells     = which((indcells_icpca != 0)[sel_rows, ]),
  rowlabels    = sel_names,
  columnlabels = colnames(X),
  mTitle       = "ICPCA \u2013 Residual map"
)
dev.off()
cat("Saved: residual_map_icpca.pdf\n")

# =============================================================================
# 8. Outlier maps
# =============================================================================
make_outlier_map <- function(SD, OD, cSD, cOD, labels, title_str,
                             label_these = NULL) {
  type <- dplyr::case_when(
    SD > cSD & OD > cOD  ~ "Bad leverage point",
    SD > cSD & OD <= cOD ~ "Good leverage point",
    SD <= cSD & OD > cOD ~ "Orthogonal outlier",
    TRUE                  ~ "Regular"
  )
  df <- data.frame(SD, OD, type, label = labels, stringsAsFactors = FALSE)
  if (is.null(label_these)) label_these <- labels[type != "Regular"]
  df$show_label <- df$label %in% label_these
  ggplot(df, aes(x = SD, y = OD, colour = type)) +
    geom_point(aes(shape = type), size = 1.8, alpha = 0.75) +
    geom_vline(xintercept = cSD, linetype = "dashed", colour = "grey50") +
    geom_hline(yintercept = cOD, linetype = "dashed", colour = "grey50") +
    geom_text_repel(data = subset(df, show_label),
      aes(label = label), size = 2.8, max.overlaps = 20, segment.size = 0.3) +
    scale_colour_manual(values = c(
      "Regular"             = "#AAAAAA", "Good leverage point" = "#4C8BB5",
      "Orthogonal outlier"  = "#E07B54", "Bad leverage point"  = "#D62728")) +
    scale_shape_manual(values = c(
      "Regular" = 1, "Good leverage point" = 2,
      "Orthogonal outlier" = 16, "Bad leverage point" = 17)) +
    labs(title = title_str, x = "Score distance (SD)",
         y = "Orthogonal distance (OD)", colour = NULL, shape = NULL) +
    theme_bw(base_size = 12) + theme(legend.position = "bottom")
}

OD_icpca  <- sqrt(rowSums((X_naimp_icpca - Xhat_icpca)^2, na.rm = TRUE))
SD_icpca  <- sqrt(rowSums(sweep(fit_icpca$scores^2, 2, fit_icpca$eigenvalues, "/")))
cSD_icpca <- sqrt(qchisq(0.99, df = k))
od23      <- OD_icpca^(2/3)
cOD_icpca <- (median(od23, na.rm=TRUE) + mad(od23, na.rm=TRUE) * qnorm(0.99))^(3/2)

p_om_macro <- make_outlier_map(fit_macro$SD, fit_macro$OD,
  fit_macro$cutoffSD, fit_macro$cutoffOD, obs_names,
  "MacroPCA \u2013 Outlier map")
p_om_icpca <- make_outlier_map(SD_icpca, OD_icpca, cSD_icpca, cOD_icpca,
  obs_names, "ICPCA \u2013 Outlier map")

p_panel <- gridExtra::grid.arrange(p_om_icpca, p_om_macro, ncol = 2)
ggsave("outlier_map_panel.pdf", p_panel, width = 14, height = 6)
cat("Saved: outlier_map_panel.pdf\n")

# =============================================================================
# 9. Loadings comparison
# =============================================================================
load_df <- data.frame(
  variable     = colnames(X),
  PC1_MacroPCA = fit_macro$loadings[, 1],
  PC2_MacroPCA = fit_macro$loadings[, 2],
  PC1_ICPCA    = fit_icpca$loadings[, 1],
  PC2_ICPCA    = fit_icpca$loadings[, 2],
  stringsAsFactors = FALSE
)
load_long <- load_df %>%
  pivot_longer(-variable, names_to = "key", values_to = "loading") %>%
  mutate(PC = sub("^(PC[0-9]+)_.*$", "\\1", key),
         method = sub("^PC[0-9]+_(.*)$", "\\1", key))

p_load <- ggplot(load_long, aes(x = variable, y = loading, fill = method)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85, width = 0.7) +
  geom_hline(yintercept = 0, colour = "grey30", linewidth = 0.4) +
  scale_fill_manual(values = c("MacroPCA" = "#2CA02C", "ICPCA" = "#E07B54")) +
  facet_wrap(~PC, ncol = 1) +
  labs(title = "Loadings: ICPCA vs MacroPCA (Global Economic Indicators 2024, k=2)",
       x = NULL, y = "Loading", fill = "Method") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")
ggsave("loadings_comparison.pdf", p_load, width = 10, height = 7)
cat("Saved: loadings_comparison.pdf\n")

# =============================================================================
# 10. Online prediction
# =============================================================================
cat("\n=== Online prediction ===\n")
all_rows   <- seq_len(nrow(X))
train_rows <- setdiff(all_rows, sel_rows)

X_tr <- X[train_rows, ]
rownames(X_tr) <- seq_len(nrow(X_tr))
X_te <- X[sel_rows, , drop = FALSE]
rownames(X_te) <- seq_len(nrow(X_te))

cat("Training on", nrow(X_tr), "countries, predicting", nrow(X_te), "\n")

fit_train <- tryCatch(
  MacroPCA(X_tr, k = k),
  error = function(e) { cat("Train fit failed:", conditionMessage(e), "\n"); fit_macro }
)

pred_list <- lapply(seq_len(nrow(X_te)), function(i) {
  tryCatch(
    MacroPCApredict(Xnew = X_te[i, , drop = FALSE], InitialMacroPCA = fit_train),
    error = function(e) NULL
  )
})

d_cols <- ncol(X)
stdR_pred     <- do.call(rbind, lapply(pred_list, function(r) {
  if (!is.null(r) && !is.null(r$stdResid)) as.numeric(r$stdResid)
  else rep(NA_real_, d_cols)
}))
indcells_pred <- matrix(0L, nrow = length(pred_list), ncol = d_cols)
for (i in seq_along(pred_list)) {
  r <- pred_list[[i]]
  if (!is.null(r) && !is.null(r$indcells) && length(r$indcells) == d_cols)
    indcells_pred[i, ] <- as.integer(r$indcells != 0)
}
colnames(stdR_pred) <- colnames(X)
rownames(stdR_pred) <- sel_names

n_ok <- sum(rowSums(!is.na(stdR_pred)) > 0)
if (n_ok > 0) {
  pdf("online_prediction_comparison.pdf", width = 14, height = 7)
  par(mfrow = c(1, 2))
  cellMap(fit_macro$stdResid[sel_rows, ], indcells = which((fit_macro$indcells != 0)[sel_rows, ]),
    rowlabels = sel_names, columnlabels = colnames(X), mTitle = "In-sample")
  cellMap(stdR_pred, indcells = which((indcells_pred != 0)),
    rowlabels = sel_names, columnlabels = colnames(X), mTitle = "Out-of-sample")
  dev.off()
  cat("Saved: online_prediction_comparison.pdf\n")
} else {
  cat("NOTE: No predictions available, skipping online prediction plot.\n")
}

# =============================================================================
# 11. Flagged countries summary
# =============================================================================
macro_type <- dplyr::case_when(
  fit_macro$SD > fit_macro$cutoffSD & fit_macro$OD > fit_macro$cutoffOD ~ "Bad leverage point",
  fit_macro$SD > fit_macro$cutoffSD  ~ "Good leverage point",
  fit_macro$OD > fit_macro$cutoffOD  ~ "Orthogonal outlier",
  TRUE                               ~ "Regular"
)
flagged_df <- data.frame(
  Country = obs_names, SD = round(fit_macro$SD, 2), OD = round(fit_macro$OD, 2),
  Type = macro_type, stringsAsFactors = FALSE
) %>% filter(Type != "Regular") %>% arrange(desc(OD))

cat("\n=== MacroPCA: Flagged countries ===\n")
print(flagged_df)
cat("\n=== Real data analysis complete ===\n")


In [ ]:
# =============================================================================
# 03_contamination_analysis.R
#
# Contamination analysis on Global Economic Indicators (2024 cross-section).
# Mirrors the spirit of Hubert et al. (2019) Section 5:
#   10% cellwise + 10% rowwise outliers + 5% extra NAs
# =============================================================================

library(cellWise)
library(ggplot2)
library(ggrepel)
library(dplyr)
library(gridExtra)
library(readr)

set.seed(2025)

# =============================================================================
# 0. Load data — same logic as cell 3
# =============================================================================
econ_raw <- NULL
csv_paths <- c(
  "/kaggle/input/datasets/giljorge/copy-world-bank-data/world_bank_development_indicators.csv"
)
for (p in csv_paths) {
  if (file.exists(p)) { econ_raw <- read_csv(p, show_col_types=FALSE); break }
}
if (is.null(econ_raw)) stop("Dataset not found.")

country_col <- colnames(econ_raw)[grepl("(?i)country", colnames(econ_raw), perl=TRUE)][1]
exclude_pat <- "(?i)(year|date|country|iso|code|region|income|name)"
num_cols    <- colnames(econ_raw)[sapply(econ_raw, is.numeric)]
num_cols    <- num_cols[!grepl(exclude_pat, num_cols)]

econ_agg <- econ_raw %>%
  group_by(!!sym(country_col)) %>%
  summarise(across(all_of(num_cols), ~ mean(.x, na.rm=TRUE)), .groups="drop")
econ_agg[num_cols] <- lapply(econ_agg[num_cols], function(x) ifelse(is.nan(x), NA, x))

na_frac   <- sapply(econ_agg[num_cols], function(x) mean(is.na(x)))
keep_cols <- num_cols[na_frac < 0.5]
if (length(keep_cols) < 2) stop("Fewer than 2 usable columns.")

X_raw <- data.matrix(econ_agg[, keep_cols, drop=FALSE])
rownames(X_raw) <- make.unique(as.character(econ_agg[[country_col]]))
X_raw <- X_raw[rowMeans(is.na(X_raw)) < 0.5, ]
obs_names <- rownames(X_raw)


# Log-transform right-skewed variables (same as cell 3)
log_candidates <- colnames(X_raw)[grepl("(?i)(gdp|trade|export|import|debt|population|fdi)",
                                         colnames(X_raw), perl=TRUE)]
X_clean <- X_raw
for (v in log_candidates) {
  col <- X_raw[, v]
  if (all(col[!is.na(col)] > 0)) X_clean[, v] <- log(col)
}
X_clean[!is.finite(X_clean)] <- NA

n <- nrow(X_clean); p <- ncol(X_clean)
cat("=== Global Economic Indicators: n =", n, ", p =", p,
    ", NAs =", sum(is.na(X_clean)), "===\n\n")

# =============================================================================
# 1. Ground-truth fits on clean data
# =============================================================================
cat("Fitting ground-truth models on clean data...\n")
fit_clean_macro <- MacroPCA(X_clean, k = k)

# Build proper indcells matrix for clean fit
clean_ind_mat <- matrix(0L, nrow(fit_clean_macro$stdResid), ncol(fit_clean_macro$stdResid))
clean_ind_mat[!is.na(fit_clean_macro$stdResid) & fit_clean_macro$stdResid >  2.576] <-  1L
clean_ind_mat[!is.na(fit_clean_macro$stdResid) & fit_clean_macro$stdResid < -2.576] <- -1L
fit_clean_macro$indcells <- clean_ind_mat

# Impute for ICPCA
X_clean_imp <- X_clean
for (j in seq_len(p)) {
  col_med <- median(X_clean_imp[, j], na.rm=TRUE)
  if (!is.finite(col_med)) col_med <- 0
  X_clean_imp[is.na(X_clean_imp[, j]), j] <- col_med
}
set.seed(42)
for (j in seq_len(p)) {
  if (sd(X_clean_imp[, j]) < 1e-10)
    X_clean_imp[, j] <- X_clean_imp[, j] + rnorm(n, 0, 1e-6)
}
fit_clean_icpca <- tryCatch(ICPCA(X_clean_imp, k=k), error=function(e) NULL)
if (is.null(fit_clean_icpca)) stop("ICPCA on clean data failed.")

# (k+1)-th direction for rowwise contamination
fit_clean_k3 <- tryCatch(MacroPCA(X_clean, k=k+1), error=function(e) NULL)
if (!is.null(fit_clean_k3) && ncol(fit_clean_k3$loadings) >= k+1) {
  v_orth <- fit_clean_k3$loadings[, k+1]
} else {
  v_orth <- fit_clean_macro$loadings[, k]   # fallback
}
center_X   <- fit_clean_macro$center
scores_orth <- (X_clean - matrix(center_X, n, p, byrow=TRUE)) %*% v_orth
orth_scale  <- mad(scores_orth, na.rm=TRUE)
if (!is.finite(orth_scale) || orth_scale < 1e-10) orth_scale <- 1
cat("Orthogonal-direction scale (mad):", round(orth_scale, 4), "\n\n")

subspace_angle <- function(P1, P2) {
  proj1 <- P1 %*% solve(t(P1) %*% P1) %*% t(P1)
  proj2 <- P2 %*% solve(t(P2) %*% P2) %*% t(P2)
  norm(proj1 - proj2, type="F") / sqrt(2 * ncol(P1))
}

# =============================================================================
# 2. Contamination
# =============================================================================
contaminate_data <- function(X, frac_cell=0.10, frac_row=0.10,
                              frac_miss=0.05, shift_cell=8, shift_row=10) {
  Xc         <- X
  true_cells <- matrix(FALSE, nrow(X), ncol(X))
  true_rows  <- rep(FALSE, nrow(X))
  center_X   <- apply(X, 2, median, na.rm=TRUE)
  col_mads   <- apply(X, 2, mad,    na.rm=TRUE)
  col_mads[col_mads < 1e-10] <- 1

  # Rowwise
  bad_rows <- sample(seq_len(nrow(X)), round(frac_row * nrow(X)))
  for (i in bad_rows) {
    Xc[i, ]      <- center_X + shift_row * orth_scale * v_orth +
                    rnorm(ncol(X)) * 0.5 * col_mads
    true_rows[i] <- TRUE
  }

  # Cellwise (non-rowwise, observed cells only)
  eligible <- which(!true_rows & matrix(TRUE, nrow(X), ncol(X)) & !is.na(X))
  n_cell   <- min(round(frac_cell * length(X)), length(eligible))
  sel      <- sample(eligible, n_cell)
  signs    <- sample(c(-1L,1L), length(sel), replace=TRUE)
  col_idx  <- ceiling(sel / nrow(X))
  row_idx  <- ((sel-1L) %% nrow(X)) + 1L
  for (ii in seq_along(sel)) {
    r <- row_idx[ii]; cc <- col_idx[ii]
    Xc[r, cc]         <- center_X[cc] + signs[ii] * shift_cell * col_mads[cc]
    true_cells[r, cc] <- TRUE
  }

  # Extra missing
  avail <- which(!is.na(Xc))
  if (length(avail) > 0) {
    miss_idx <- sample(avail, min(round(frac_miss*length(X)), length(avail)))
    Xc[miss_idx] <- NA
  }
  list(X_cont=Xc, true_cells=true_cells, true_rows=true_rows)
}

cont   <- contaminate_data(X_clean)
X_cont <- cont$X_cont

cat("=== Contamination summary ===\n")
cat("  Cellwise outliers:", sum(cont$true_cells),
    sprintf("cells (%.1f%%)\n", 100*mean(cont$true_cells)))
cat("  Rowwise outliers: ", sum(cont$true_rows),
    sprintf("rows  (%.1f%%)\n", 100*mean(cont$true_rows)))
cat("  Missing values:   ", sum(is.na(X_cont)),
    sprintf("cells (%.1f%%)\n\n", 100*mean(is.na(X_cont))))

# =============================================================================
# 3. Fit methods on contaminated data
# =============================================================================
cat("Fitting MacroPCA on contaminated data...\n")
fit_cont_macro <- MacroPCA(X_cont, k=k)

# BUILD indcells as proper matrix — THIS was the crash in heressss.ipynb
cont_ind_mat <- matrix(0L, nrow(fit_cont_macro$stdResid), ncol(fit_cont_macro$stdResid))
cont_ind_mat[!is.na(fit_cont_macro$stdResid) & fit_cont_macro$stdResid >  2.576] <-  1L
cont_ind_mat[!is.na(fit_cont_macro$stdResid) & fit_cont_macro$stdResid < -2.576] <- -1L
fit_cont_macro$indcells <- cont_ind_mat

# =============================================================================
# 4. Alignment: identify rows MacroPCA actually analyzed
# =============================================================================
kept_macro <- which(rowMeans(is.na(X_cont)) <= 0.5)
n_kept     <- length(kept_macro)
cat("\nAlignment: MacroPCA analyzed", n_kept, "of", n, "rows\n")
dropped <- setdiff(seq_len(n), kept_macro)
if (length(dropped) > 0) cat("Dropped rows (>50% NA):", dropped, "\n")

OD_full       <- rep(NA_real_, n)
SD_full       <- rep(NA_real_, n)
indrows_full  <- rep(FALSE, n)
indcells_full <- matrix(0L, n, p)
stdResid_full <- matrix(NA_real_, n, p)

OD_full[kept_macro]         <- fit_cont_macro$OD
SD_full[kept_macro]         <- fit_cont_macro$SD
stdResid_full[kept_macro, ] <- fit_cont_macro$stdResid
indcells_full[kept_macro, ] <- fit_cont_macro$indcells

ir_vec <- fit_cont_macro$indrows
if (length(ir_vec) == n_kept) indrows_full[kept_macro] <- as.logical(ir_vec)

# ICPCA: fit only on the same rows MacroPCA used, to keep dimensions aligned
cat("Fitting ICPCA on contaminated data...\n")
X_cont_sub <- X_cont[kept_macro, ]           # n_kept x p
X_cont_imp <- X_cont_sub
for (j in seq_len(p)) {
  col_med <- median(X_cont_imp[, j], na.rm=TRUE)
  if (!is.finite(col_med)) col_med <- 0
  X_cont_imp[is.na(X_cont_imp[, j]), j] <- col_med
}
lo <- quantile(X_cont_imp, 0.001, na.rm=TRUE)
hi <- quantile(X_cont_imp, 0.999, na.rm=TRUE)
X_cont_imp[X_cont_imp < lo] <- lo
X_cont_imp[X_cont_imp > hi] <- hi
for (j in seq_len(p)) {
  if (sd(X_cont_imp[, j]) < 1e-10)
    X_cont_imp[, j] <- X_cont_imp[, j] + rnorm(nrow(X_cont_imp), 0, 1e-6)
}
fit_cont_icpca <- tryCatch(ICPCA(X_cont_imp, k=k), error=function(e) {
  message("ICPCA (contaminated) failed: ", conditionMessage(e)); NULL
})

# Expand ICPCA predictions back to full n rows
Xhat_icpca_full <- matrix(NA_real_, n, p)
if (!is.null(fit_cont_icpca)) {
  Xhat_kept <- sweep(fit_cont_icpca$scores %*% t(fit_cont_icpca$loadings),
                     2, fit_cont_icpca$center, "+")
  Xhat_icpca_full[kept_macro, ] <- Xhat_kept   # n_kept rows -> correct
}
X_naimp_icpca <- X_cont
for (j in seq_len(p)) {
  na_j <- is.na(X_cont[, j]) & !is.na(Xhat_icpca_full[, j])
  X_naimp_icpca[na_j, j] <- Xhat_icpca_full[na_j, j]
}

# =============================================================================
# 5. Subspace recovery
# =============================================================================
angle_macro <- subspace_angle(fit_cont_macro$loadings, fit_clean_macro$loadings)
angle_icpca <- if (!is.null(fit_cont_icpca))
  subspace_angle(fit_cont_icpca$loadings, fit_clean_icpca$loadings) else NA_real_

cat("\n=== Subspace recovery (lower = better) ===\n")
cat(sprintf("  ICPCA    angle: %s\n", if(is.na(angle_icpca)) "NA" else round(angle_icpca,4)))
cat(sprintf("  MacroPCA angle: %.4f\n", angle_macro))
if (!is.na(angle_icpca)) {
  impr <- 100*(angle_icpca - angle_macro)/max(angle_icpca, 1e-10)
  cat(sprintf("  Improvement:    %.1f%%\n\n", impr))
}

# =============================================================================
# 6. Detection evaluation
# =============================================================================
eval_rows <- kept_macro
detected_rows <- indrows_full
true_rows_v   <- cont$true_rows

TP_r <- sum( detected_rows[eval_rows] &  true_rows_v[eval_rows])
FP_r <- sum( detected_rows[eval_rows] & !true_rows_v[eval_rows])
FN_r <- sum(!detected_rows[eval_rows] &  true_rows_v[eval_rows])
TN_r <- sum(!detected_rows[eval_rows] & !true_rows_v[eval_rows])
sens_row <- TP_r / max(TP_r+FN_r, 1)
spec_row <- TN_r / max(TN_r+FP_r, 1)
prec_row <- if ((TP_r+FP_r) > 0) TP_r/(TP_r+FP_r) else NA_real_
f1_row   <- if (!is.na(prec_row) && (prec_row+sens_row) > 0)
              2*prec_row*sens_row/(prec_row+sens_row) else NA_real_

cat("=== Rowwise detection (MacroPCA) ===\n")
cat("  TP:", TP_r, " FP:", FP_r, " FN:", FN_r, "\n")
cat(sprintf("  Sensitivity: %.3f | Specificity: %.3f\n", sens_row, spec_row))

eval_mask      <- matrix(FALSE, n, p); eval_mask[kept_macro, ] <- TRUE
observed       <- !is.na(X_cont)
true_c_eval    <- cont$true_cells & observed & eval_mask
detected_c_eval <- (indcells_full != 0) & observed & eval_mask

TP_c <- sum( detected_c_eval &  true_c_eval)
FP_c <- sum( detected_c_eval & !true_c_eval)
FN_c <- sum(!detected_c_eval &  true_c_eval)
sens_cell <- TP_c / max(TP_c+FN_c, 1)
prec_cell <- if ((TP_c+FP_c)>0) TP_c/(TP_c+FP_c) else NA_real_
f1_cell   <- if (!is.na(prec_cell) && (prec_cell+sens_cell)>0)
               2*prec_cell*sens_cell/(prec_cell+sens_cell) else NA_real_

cat("=== Cellwise detection (MacroPCA) ===\n")
cat("  TP:", TP_c, " FP:", FP_c, " FN:", FN_c, "\n")
cat(sprintf("  Sensitivity: %.3f | Precision: %s | F1: %s\n\n",
    sens_cell,
    if(is.na(prec_cell)) "NA" else round(prec_cell,3),
    if(is.na(f1_cell))   "NA" else round(f1_cell,3)))

# =============================================================================
# 7. Residual maps
# =============================================================================
true_any  <- which(cont$true_rows | rowSums(cont$true_cells) > 0)
top_od    <- kept_macro[order(OD_full[kept_macro], decreasing=TRUE)][1:min(10,n_kept)]
disp_rows <- intersect(unique(c(true_any, top_od)), kept_macro)
disp_rows <- disp_rows[seq_len(min(24, length(disp_rows)))]
disp_names   <- obs_names[disp_rows]
disp_in_kept <- match(disp_rows, kept_macro)

pdf("contamination_residual_map_macropca.pdf", width=12, height=7)
local({
  R_sub <- fit_cont_macro$stdResid[disp_in_kept, ]
  I_sub <- fit_cont_macro$indcells[disp_in_kept, ]
  cellMap(
    R_sub,
    indcells     = which(I_sub != 0),
    rowlabels    = disp_names, columnlabels = colnames(X_clean),
    mTitle       = "MacroPCA \u2013 Contaminated data residual map"
  )
})
dev.off()

resid_icpca_c <- X_naimp_icpca - Xhat_icpca_full
col_mad_c     <- apply(resid_icpca_c, 2, function(x) mad(x, na.rm=TRUE))
col_mad_c[col_mad_c < 1e-10] <- 1
stdR_icpca_c  <- sweep(resid_icpca_c, 2, col_mad_c, "/")
indc_icpca_c  <- matrix(0L, n, p)
indc_icpca_c[!is.na(stdR_icpca_c) & stdR_icpca_c >  2.576] <-  1L
indc_icpca_c[!is.na(stdR_icpca_c) & stdR_icpca_c < -2.576] <- -1L

pdf("contamination_residual_map_icpca.pdf", width=12, height=7)
local({
  R_sub <- stdR_icpca_c[disp_rows, ]
  I_sub <- indc_icpca_c[disp_rows, ]
  cellMap(
    R_sub,
    indcells     = which(I_sub != 0),
    rowlabels    = disp_names, columnlabels = colnames(X_clean),
    mTitle       = "ICPCA \u2013 Contaminated data residual map"
  )
})
dev.off()
cat("Saved: contamination residual maps\n")

# =============================================================================
# 8. Outlier map coloured by ground truth
# =============================================================================
truth_label <- dplyr::case_when(
  cont$true_rows                   ~ "True rowwise outlier",
  rowSums(cont$true_cells) > 0     ~ "True cellwise outlier",
  TRUE                             ~ "Regular"
)
plot_df <- data.frame(
  SD=SD_full[kept_macro], OD=OD_full[kept_macro],
  Truth=truth_label[kept_macro], label=obs_names[kept_macro],
  stringsAsFactors=FALSE
)
p_om <- ggplot(plot_df, aes(x=SD, y=OD, colour=Truth)) +
  geom_point(size=1.8, alpha=0.7) +
  geom_vline(xintercept=fit_cont_macro$cutoffSD, linetype="dashed", colour="grey40") +
  geom_hline(yintercept=fit_cont_macro$cutoffOD, linetype="dashed", colour="grey40") +
  geom_text_repel(
    data=subset(plot_df, Truth!="Regular"|SD>fit_cont_macro$cutoffSD|OD>fit_cont_macro$cutoffOD),
    aes(label=label), size=2.5, max.overlaps=15) +
  scale_colour_manual(values=c("Regular"="#BBBBBB",
    "True rowwise outlier"="#D62728","True cellwise outlier"="#4C8BB5")) +
  labs(title="MacroPCA outlier map \u2013 contaminated Economic Indicators",
       subtitle="Coloured by ground-truth contamination label",
       x="Score distance (SD)", y="Orthogonal distance (OD)", colour="Ground truth") +
  theme_bw(base_size=12) + theme(legend.position="bottom")
ggsave("contamination_outlier_map.pdf", p_om, width=7, height=6)
cat("Saved: contamination_outlier_map.pdf\n")

# =============================================================================
# 9. Summary charts
# =============================================================================
angle_df <- data.frame(
  Method = c("ICPCA","MacroPCA"),
  Angle  = c(ifelse(is.na(angle_icpca),0,angle_icpca), angle_macro)
)
p_angle <- ggplot(angle_df, aes(x=Method, y=Angle, fill=Method)) +
  geom_bar(stat="identity", width=0.5, alpha=0.85) +
  geom_text(aes(label=round(Angle,4)), vjust=-0.4, size=4) +
  scale_fill_manual(values=c("ICPCA"="#E07B54","MacroPCA"="#2CA02C"), guide="none") +
  labs(title="Subspace recovery on contaminated data",
       subtitle="Lower angle = closer to clean-data reference",
       x=NULL, y="Subspace angle") + theme_bw(base_size=12)

perf_df <- data.frame(
  Metric = c("Sensitivity","Specificity","Precision","F1",
             "Sensitivity","Precision","F1"),
  Value  = c(sens_row, spec_row,
             ifelse(is.na(prec_row),0,prec_row),
             ifelse(is.na(f1_row),0,f1_row),
             sens_cell,
             ifelse(is.na(prec_cell),0,prec_cell),
             ifelse(is.na(f1_cell),0,f1_cell)),
  Type   = c(rep("Rowwise",4), rep("Cellwise",3))
)
p_perf <- ggplot(perf_df, aes(x=Metric, y=Value, fill=Type)) +
  geom_bar(stat="identity", position="dodge", alpha=0.85, width=0.65) +
  geom_text(aes(label=sprintf("%.2f",Value)),
            position=position_dodge(width=0.65), vjust=-0.4, size=3.5) +
  scale_fill_manual(values=c("Rowwise"="#D62728","Cellwise"="#4C8BB5")) +
  scale_y_continuous(limits=c(0,1.15), breaks=seq(0,1,0.2)) +
  labs(title="Detection performance (MacroPCA)",
       subtitle="10% rowwise + 10% cellwise + 5% extra NAs",
       x=NULL, y="Rate", fill="Outlier type") +
  theme_bw(base_size=12) + theme(legend.position="bottom")

p_sum <- gridExtra::grid.arrange(p_angle, p_perf, ncol=2)
ggsave("contamination_summary.pdf", p_sum, width=12, height=5)
cat("Saved: contamination_summary.pdf\n")

cat("\n=================================================================\n")
cat("KEY FINDINGS\n=================================================================\n")
cat(sprintf("\nSUBSPACE RECOVERY\n"))
cat(sprintf("  ICPCA    angle = %s\n", if(is.na(angle_icpca)) "NA" else round(angle_icpca,4)))
cat(sprintf("  MacroPCA angle = %.4f\n", angle_macro))
cat("\nROWWISE DETECTION\n")
cat(sprintf("  Sens %.2f | Spec %.2f | Prec %s | F1 %s\n",
    sens_row, spec_row,
    if(is.na(prec_row)) "NA" else sprintf("%.2f",prec_row),
    if(is.na(f1_row))   "NA" else sprintf("%.2f",f1_row)))
cat("\nCELLWISE DETECTION\n")
cat(sprintf("  Sens %.2f | Prec %s | F1 %s\n",
    sens_cell,
    if(is.na(prec_cell)) "NA" else sprintf("%.2f",prec_cell),
    if(is.na(f1_cell))   "NA" else sprintf("%.2f",f1_cell)))
cat("=================================================================\n")
